# Foreword
---
This notebook and calculations are written by Erik Leusink on behalf of the Energy Transition Lectorate. The calculations are based on the data provided by the Energy Transition Lectorate. The calculations are made to provide insight into the heat loss coefficient of a house. The calculations are made using linear regression. Everything in this notebook is documented and written in English to make it accessible for everyone who is interested in the calculations and the results.

# Abstract
---
This notebook calculates the heat loss coefficient of a house using linear regression. To calculate the heat loss coefficient, the following steps are taken:

# Business understanding
The Energy Trasition lectorate are trying to make houses more sustainable. Their focus is to reduce the climate change as fast as possible with solutions mostly focused on the time before 2050. They focus on the biggest group of houses in this time period which are existing buildings mostly because the building of new houses is only a small portion of the houses in 2050. 

The challenge of this project is to give homeowners insight into their energy consumption and how much money they can save by making their house more sustainable. The goal is to make the homeowner aware of the heat loss and the money they can save by making their house more sustainable. Houses do already have energy labels, but these labels are not always accurate. We want to make a more accurate prediction of the heat loss coefficient of a house.

### Why are we focussing on existing buildings?
The focus on existing buildings is due to the fact that they make up the majority of the housing stock that will be present in 2050. The construction of new houses is only a small portion of the houses in 2050. Therefore, making existing buildings more sustainable can have a larger impact on reducing climate change.

### Why are we focusing on heat loss?
The heat is one of the biggest energy consumers in a house. It's almost 50 % of the energy consumption in a house. All the heat that gets lost in a house is heat that needs to be filled up again using energy. With focusing on the heat loss we can make the biggest impact on the energy consumption of a house. Also the heat loss coefficient is a good indicator of how sustainable a house is. The lower the heat loss coefficient the more sustainable a house is.

### What is the heat loss coefficient?
The heat loss coefficient is the amount of energy that is lost through the walls, roof, windows, and doors of a house. The heat loss coefficient is measured in W/K. The naming of the heat loss is a bit deceiving even tough it's called loss it can also work the other way around when the outside temperature is higher than the inside temperature. So in simpele words it's how easy heat can get in or out of a house.

### Does this question add value to the Energy Transition Lectorate?
Yes, this question adds value to the Energy Transition Lectorate. The Energy Transition Lectorate wants to make houses more sustainable. By giving homeowners insight into their energy consumption and how much money they can save by making their house more sustainable, the Energy Transition Lectorate can make a big impact on the energy consumption of a house. The heat loss coefficient is a good indicator of how sustainable a house is.

### How should we measure the success of this project?
The success of this project should be measured by the accuracy of the prediction of the heat loss coefficient of a house. Eventualy the prediction should be more accurate than the current energy labels. When the overall predictions are at least 80% accurate we can use this method effectivly. The lectorate wanted to know if using static models is accurate enough for finding the heat loss coefficient. Using static models is much less intensive for computer calculation and can be more cost effective.

# Data understanding
We have three datasets. Two of the datasets contain 20 houses which are identified with a pseudo anonymized number. 
The first dataset (need_for_heat) contains the gas meter reading and the inside temperature of the house.
The second dataset (remeha) contains the gas usage domestic hot water and the gas usage for heating. 
The third dataset (weather) contains the:
   - T: Temperature
   - FH: Hourly average wind speed
   - DD: Wind direction
   - Q: Global radiation
   - DR: Duration of precipitation
   - RH: Hourly amount of precipitation
   - N: Cloudy
   - U: Relative humidity (in percent)" with timestamps.

The data is collected from the houses spread over the Netherlands and the data is collected over a period from 1st of December 2023 till around the end of March 2024. Sometimes the data is missing because of technical issues or alot of 0 values because the house was not occupied. The data is collected around every 10 minutes.

The worst houses are removed in the beginnen of the notebooks which are the houses with almost no readings.

The objective is to have a model that can predict the heat loss coefficient of a house. The heat loss coefficient is the amount of energy that is lost through the walls, roof, windows, and doors of a house. The heat loss coefficient is measured in W/K. The heat loss coefficient is calculated by the following formula:
We first get the gas usage using the difference in gas meter readings. Then we calculate the amount of heat gained by burning gas. It's important to know what the calorific heating value is of the gas and for the efficiency of the boiler we get the mean value. 
This results in heat_gain_ch__J = Σ(g_use__m3 * h_sup__MJ_m_3 * 1000000 * eta_sup__J0)
where g_use is gas used, h_sup is the superior calorific heating value of Groningen gas and the eta_sup is the mean efficiency of the boiler in the Netherlands (source: WoON2008; ISSO 82.3)

To find out how much heating is needed and see how efficient a house is we need to know the temperature demand.
We will subtract the inside temperature from the outside times the interval so we know how much temperature is demanded.
temp_demand__ K_s = Σ(temp_in__degC - temp_out__degC) * interval__s
Where temp_in is temperature inside, temp_out is the outside temperature. interval is time between datapoints

We can calculate/predict the H__W_K_1 value by using the temp_demand__K_s as the X-Value and the Heat_gain_ch__J as the Y-Value

---
### The following steps: (Data preperation, Modeling, Evalutaion) are going back and forth. Some insights in the Modeling can require an action in the Data preperation.

# Data preparation
In this notebook the original data is not altered but only filtered in the process. Mainly because some houses has almost no data, or measurements are not correct which are detected uisng boxplots and Tukey fences. Sometimes the outliers are removed. Sometimes it is not possible due to the large amount of 0-values which outweigh almost every measurement.

All the dataframes are eventually merged into one dataframe for the modeling. For the inside temperature we used the value of the remeha dataset. This one is more accurate with the temperature. The gas usage is calculated by the difference in gas meter readings. So we use the need_for_heat dataset for this. The weather dataset is used for the outside temperature.

Eventually the Dataframe is grouped per day so the meter values are summed per day. The measurements are per 10 minutes and the difference between the measurements which result in alot of 0-values 

# Modeling

In [ ]:
#Importing all the necessary libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import sys
from HourlyHistoricWeather import historicdutchweather
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
sys.path.append('HourlyHistoricWeather')

In [ ]:
#Loading the datasets to dataframes
needforheat_file_path='./Data/needforheat_export_student.parquet'
remeha_file_path='./Data/remeha_export_student.parquet'
weather_file_path='./Data/weather_interpolation_locations.parquet'

In [ ]:
try:
    df_n= pd.read_parquet(
        needforheat_file_path,
        engine='pyarrow',
        #use_nullable_dtypes=True #Outcommented because of futureWarning
        dtype_backend='numpy_nullable'
    )
    print("Need_for_heat dataset was successfully read without specifying compression codec.")
    
    df_r= pd.read_parquet(
        remeha_file_path,
        engine='pyarrow',
        #use_nullable_dtypes=True #Outcommented because of futureWarning
        dtype_backend='numpy_nullable'
    )
    print("Remeha dataset was successfully read without specifying compression codec.")
    
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
#Loading the weather data from the HourlyHistoricWeather package
x = datetime(2024,1,1)
y = datetime(2024,3,31)
lat, lon = 52.50938258156131, 6.1113935843428715

df_w = historicdutchweather.get_local_weather(x, y, lat, lon)

In [ ]:
# Making copies of all the dataframes for rerunning the notebook without reloading the data
df_nfh = df_n.copy()
df_remeha = df_r.copy()
df_weather = df_w.copy()

In [ ]:
df_nfh.reset_index(inplace=True)
df_remeha.reset_index(inplace=True)
df_weather.reset_index(inplace=True)

# Removing the worst houses with almost no readings
df_nfh = df_nfh[df_nfh['id'] != 591006]
df_nfh = df_nfh[df_nfh['id'] != 558926]
df_nfh = df_nfh[df_nfh['id'] != 504607]
df_nfh = df_nfh[df_nfh['source_category'] == 'device']

In [ ]:
# Relocating the properties and values to their own columns
df_nfh['gas_usage_cum__m3'] = df_nfh[df_nfh['property'] == 'g_use_cum__m3']['value']
df_nfh['T_inside__degC'] = df_nfh[df_nfh['property'] == 'temp_in__degC']['value']
df_nfh.drop(columns=['property', 'value'], inplace=True)
df_remeha['g_use_dhw_inf_cum__kWh'] = df_remeha[df_remeha['property'] == 'g_use_dhw_inf_cum__kWh']['value']
df_remeha['g_use_ch_inf_cum__kWh'] = df_remeha[df_remeha['property'] == 'g_use_ch_inf_cum__kWh']['value']
df_remeha['temp_in__degC'] = df_remeha[df_remeha['property'] == 'temp_in__degC']['value']
df_remeha.drop(columns=['property', 'value'], inplace=True)

In [ ]:
# Getting the gas usage by the difference in gas meter readings
df_nfh['gas_usage__m3'] = df_nfh['gas_usage_cum__m3'].diff()
df_nfh['gas_usage__m3_filled'] = df_nfh['gas_usage__m3'].fillna(0)

In [ ]:
df_nfh.set_index('timestamp', inplace=True)

In [ ]:
df_remeha['timestamp'] = pd.to_datetime(df_remeha['timestamp'])

In [ ]:
df_remeha.set_index('timestamp', inplace=True)
df_weather.set_index('index', inplace=True)

In [ ]:
df_nfh_resampled = df_nfh.groupby('id').resample('H').agg({
    'gas_usage__m3_filled' : 'sum',
    'T_inside__degC' : 'mean'
}).reset_index()

In [ ]:
df_nfh_resampled = df_nfh_resampled.rename(columns={'gas_usage__m3_filled' : 'gas_usage__m3'})

In [ ]:
sns.boxplot(y=df_nfh_resampled['gas_usage__m3'])

In [ ]:
df_nfh_resampled = df_nfh_resampled[(df_nfh_resampled['gas_usage__m3'] > 0) | (df_nfh_resampled['gas_usage__m3'] == 0)]

In [ ]:
sns.boxplot(y=df_nfh_resampled['gas_usage__m3'])

In [ ]:
df_nfh = df_nfh_resampled

In [ ]:
df_nfh = df_nfh[df_nfh['T_inside__degC'] != 'nat']

In [ ]:
df_remeha = df_remeha.groupby('id').resample('H').agg({
    'g_use_dhw_inf_cum__kWh' : 'sum',
    'g_use_ch_inf_cum__kWh' : 'sum',
    'temp_in__degC' : 'mean'
}).reset_index()

In [ ]:
df_remeha.dropna(inplace=True)

In [ ]:
df = pd.merge(df_nfh, df_remeha, on=['id', 'timestamp'], how='inner')

In [ ]:
df = df[['id','timestamp', 'gas_usage__m3', 'g_use_ch_inf_cum__kWh', 'g_use_dhw_inf_cum__kWh', 'temp_in__degC']]

In [ ]:
h_sup__MJ_m_3 = 35.17
eta_sup_J0 = 0.97
df['heat_gain_ch__J'] = df['gas_usage__m3'] * h_sup__MJ_m_3 * 1000000 * eta_sup_J0

In [ ]:
df = df.merge(df_weather, how='left', left_on='timestamp', right_on='index')

In [ ]:
df.rename(columns={'T' : 'temp_out__degC', 'Q' : 'ghi__J_cm2'}, inplace=True)

In [ ]:
df['temp_demand__K_h'] = (df['temp_in__degC'] - df['temp_out__degC']) * 3600

In [ ]:
h_sup__MJ_m_3 = 35.17
eta_sup_J0 = 0.97

In [ ]:
df_day = df.groupby('id').resample('D', on='timestamp').agg({
    'temp_in__degC': 'mean',
    'temp_out__degC': 'mean',
    'ghi__J_cm2': 'sum',
    'temp_demand__K_h': 'sum',
    'heat_gain_ch__J': 'sum'
}).reset_index()

In [ ]:
#Renaming the columns so the correct units and metrics are used.
df_day.rename(columns={'temp_in__degC': 'temp_in_mean_d__degC', 'temp_out__degC': 'temp_out_mean_d__degC', 'ghi__J_cm2': 'ghi_sum_d__J_cm2', 'temp_demand__K_h': 'temp_demand__K_d', 'heat_gain_ch__J': 'heat_gain_ch_sum_d__J'}, inplace=True)

In [ ]:
results = []
average_r2score = 0

for house in df_day['id'].unique():
    house_data = df_day[df_day['id'] == house]
    X = house_data['temp_demand__K_d'].values.reshape(-1, 1)
    y = house_data['heat_gain_ch_sum_d__J'].values

    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)
    r2_value = r2_score(y, y_pred)

    results.append({'id': house, 'H__W_K_1': reg.coef_[0], 'R-Squared': r2_value})
    average_r2score += r2_value

    plt.scatter(X, y, label='Actual')
    plt.plot(X, y_pred, color='red', label='Predicted')
    plt.xlabel('temp_demand__K_d')
    plt.ylabel('heat_gain_ch_sum_d__J')
    plt.title(f'House ID: {house}')
    plt.legend()
    plt.show()

    print(f'House ID: {house}')
    print(f'R-Squared: {round(r2_value, 3)}')
    print(f'H__W_K_1: {round(reg.coef_[0], 3)}')

print(f'Average R-Squared: {round(average_r2score / len(df["id"].unique()), 3)}')

In [ ]:
pd.DataFrame(results)

## Using Train Test Split to validate the model

In [ ]:
results = []
average_train_r2score = 0
average_test_r2score = 0

for house in df_day['id'].unique():
    house_data = df_day[df_day['id'] == house]
    X = house_data['temp_demand__K_d'].values.reshape(-1, 1)  # Ensure data type is float64
    y = house_data['heat_gain_ch_sum_d__J'].values 

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    reg = LinearRegression().fit(X_train, y_train)

    y_train_pred = reg.predict(X_train)
    y_test_pred = reg.predict(X_test)
    
    train_r2_value = r2_score(y_train, y_train_pred)
    test_r2_value = r2_score(y_test, y_test_pred)
    
    results.append({'id': house, 'H__W_K_1': reg.coef_[0], 'Train R-Squared': train_r2_value, 'Test R-Squared': test_r2_value})
    
    average_train_r2score += train_r2_value
    average_test_r2score += test_r2_value
   
    plt.scatter(X_train, y_train, color='blue', label='Training data')
    plt.scatter(X_test, y_test, color='orange', label='Test data')
   
    plt.plot(X_test, y_test_pred, color='red', label='Predicted on test data')
    
    plt.xlabel('temp_demand__K_d')
    plt.ylabel('heat_gain_ch_sum_d__J')
    
    plt.title(f'House ID: {house}')
    plt.legend()
    plt.show()

    print(f'House ID: {house}')
    print(f'Train R-Squared: {round(train_r2_value, 3)}')
    print(f'Test R-Squared: {round(test_r2_value, 3)}')
    print(f'H__W_K_1: {round(reg.coef_[0], 3)}')
    print(f'Number of zeros in training data: {sum(y_train == 0)}')
    print(f'Number of zeros in test data: {sum(y_test == 0)}')

num_houses = len(df_day['id'].unique())
print('--------')
print(f'Average Train R-Squared: {round(average_train_r2score / num_houses, 3)}')
print(f'Average Test R-Squared: {round(average_test_r2score / num_houses, 3)}')

# Conclusion train test split model
Most of the predictions in the test data are scoring higher than the training data. Except for the houses where there are no 0-values in the test data which result in a less steep line. There is not an opportunity to verplicht the test data to have an 0-value inside. But having the extra info underneath the model of the amount of 0-values explains the graph better.

---
## After adding the 0-value information of the train and test sets:
We can see that if the test has no 0-values the model is less steep and furter away from the actual data. But also when the train data has no 0-value because it's allocated in the test data the result is worse.

# Evaluation

My first version of the linear regression model used the data per hour. This resulted in a very low R2 score because the amount of 0-values in the measurements outweighed the real measurements. That model was not accurate enough to predict the heat loss coefficient of a house. Now on a day measurements scale the model is much more accurate (from 0.20 to 0.46 mean).

0.46 is still a bit low, but it's a good start. The model can be improved by adding more features to the model. For now I've got two hypotheses: 
- The outside temperature is not the only factor that influences the heat loss coefficient of a house. The global radiation can also influence the heat loss coefficient of a house.
- When people are not at home the heating is turned off. This can also influence the heat loss coefficient of a house. So by removing the 0-values from the day measurements the model can be improved.

# Hypotheses 1 - Global radiation
Can the global radiation influence the heat loss coefficient of a house?

We'll add the global horizontal radiation to the model and see if the R2 score improves.

In [ ]:
df_day.isna().sum()

In [ ]:
df_day.groupby('id').count()

In [ ]:
#df_day.groupby('id').mean()

In [ ]:
#df_day[df_day['id'] == 545968]

In [ ]:
# Drop the nan values for because the 
df_day.dropna(inplace=True)

In [ ]:
results = []
average_r2score = 0

for house in df_day['id'].unique():
    house_data = df_day[df_day['id'] == house]
    X = house_data[['temp_demand__K_d', 'ghi_sum_d__J_cm2']].values
    y = house_data['heat_gain_ch_sum_d__J'].values

    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)
    r2_value = r2_score(y, y_pred)

    results.append({'id': house, 'H__W_K_1': reg.coef_[0], 'R-Squared': r2_value})
    average_r2score += r2_value

    plt.scatter(X[:, 0], y, label='Actual')
    plt.scatter(X[:, 0], y_pred, color='red', label='Predicted')
    plt.xlabel('temp_demand__K_d')
    plt.ylabel('heat_gain_ch_sum_d__J')
    plt.title(f'House ID: {house}')
    plt.legend()
    plt.show()

    print(f'House ID: {house}')
    print(f'R-Squared: {round(r2_value, 3)}')
    print(f'H__W_K_1: {round(reg.coef_[0], 3)}')

print(f'Average R-Squared: {round(average_r2score / len(df_day["id"].unique()), 3)}')

# Conclusion hypothesis 1

The Average R2 score is now higher than before. This means that the global radiation can influence the heat loss coefficient of a house.

# Hypotheses 2 - Removing the 0-values

In [ ]:
df_day_no_zeros = df_day[df_day['heat_gain_ch_sum_d__J'] > 1]

In [ ]:
results = []
average_r2score = 0

for house in df_day_no_zeros['id'].unique():
    house_data = df_day_no_zeros[df_day_no_zeros['id'] == house]
    X = house_data['temp_demand__K_d'].values.reshape(-1, 1)
    y = house_data['heat_gain_ch_sum_d__J'].values

    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)
    r2_value = r2_score(y, y_pred)

    results.append({'id': house, 'H__W_K_1': reg.coef_[0], 'R-Squared': r2_value})
    average_r2score += r2_value

    plt.scatter(X, y, label='Actual')
    plt.plot(X, y_pred, color='red', label='Predicted')
    plt.xlabel('temp_demand__K_d')
    plt.ylabel('heat_gain_ch_sum_d__J')
    plt.title(f'House ID: {house}')
    plt.legend()
    plt.show()

    print(f'House ID: {house}')
    print(f'R-Squared: {round(r2_value, 3)}')
    print(f'H__W_K_1: {round(reg.coef_[0], 3)}')

print(f'Average R-Squared: {round(average_r2score / len(df["id"].unique()), 3)}')

# Conclusion hypothesis 2
Removing the 0-values from the day measurements did not improve the average model. The R2 score is lower than before. This means that the 0-values are important for the model. But when we look at house 545968 we see that the 0-values are not correct. The gas meter readings are not changing. So when there are houses with no changes in the gas meter readings, the model can be improved by removing these houses from the dataset.

# Deployment and the future

The question from the Energy Transition Lectorate is answered. The heat loss coefficient of a house can be predicted using linear regression. The model can be improved by adding more features to the model. The global radiation can influence the heat loss coefficient of a house. The 0-values are important for the model. But when there are houses with no changes in the gas meter readings, the model can be improved by removing these houses from the dataset. So it's important in the future when people want to calculate their heat loss coefficient that they keep track of the gas meter readings. Otherwise it gives an false prediction of the heat loss coefficient of a house.